## Library Import

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image, ImageOps
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import time

import tensorflow as tf


import keras
from keras import layers
from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
from sklearn.metrics import classification_report
from tensorflow.keras.callbacks import EarlyStopping
from keras.preprocessing.image import ImageDataGenerator


from sklearn.naive_bayes import MultinomialNB


import xgboost as xgb
from sklearn import svm
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

: 

## Data preprocessing

In [ ]:
# label tagging to the images

levels = ['Normal', 'COVID', 'Lung_Opacity', 'Viral_Pneumonia']
path = r"C:\Users\tejus\Desktop\projects d\Pr project - covid with xray\COVID-19_Radiography_Dataset"
data_dir = os.path.join(path)

data = []
for id, level in enumerate(levels):
    for file in os.listdir(os.path.join(data_dir, level + '/images')):
        data.append(['{}/images/{}'.format(level, file), level])

data = pd.DataFrame(data, columns = ['image_file', 'result'])

data['path'] = path + '/' + data['image_file']

data.head()

In [ ]:
print("Numbers of X-ray images: {}".format(data.shape[0]))

In [ ]:
#visualisation 

n_samples = 2

fig, m_axs = plt.subplots(4, n_samples, figsize = (6*n_samples, 3*4))

for n_axs, (type_name, type_rows) in zip(m_axs, data.sort_values(['result']).groupby('result')):
    n_axs[1].set_title(type_name, fontsize = 15)
    for c_ax, (_, c_row) in zip(n_axs, type_rows.sample(n_samples, random_state = 1234).iterrows()):       
        picture = c_row['path']
        image = cv2.imread(picture)
        c_ax.imshow(image)
        c_ax.axis('off')

In [ ]:
print(data['result'].unique())



In [ ]:
print(data[data['result'].isnull()])

In [ ]:
result_counts = data['result'].value_counts().reset_index()

plt.figure(figsize=(10, 6))
sns.barplot(x='index', y='result', data=result_counts)
plt.xticks(rotation=90)
plt.show()

In [ ]:
print('Normal : ', list(data['result']).count('Normal'))
print('Covid : ', list(data['result']).count('COVID'))
print('Opacity : ', list(data['result']).count('Lung_Opacity'))
print('Viral_Pneumonia : ', list(data['result']).count('Viral_Pneumonia'))
#10192 images sont normales

In [ ]:
round(data['result'].value_counts() / data.shape[0] * 100,2)

# Classification using pixel feature extraction 

In [ ]:
pixel_img = []

for image in tqdm(data['path']):
    img=Image.open(image)
    img=ImageOps.grayscale(img)
    img=img.resize((64,64))
    img=np.asarray(img)
    img=img.reshape((64,64,1))
    pixel_img.append(img)

In [ ]:
pixel_img = np.array(pixel_img)
label_img = data['result'].map({'Normal': 0, 'COVID': 1, 'Lung_Opacity' : 2,
                               'Viral_Pneumonia' : 3})

print(pixel_img.shape, label_img.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(pixel_img, label_img, 
                                                    test_size=0.2, stratify=label_img)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

In [ ]:
round(y_train.value_counts() / y_train.shape[0] * 100,2)

In [ ]:
input_size = X_train.shape[1] * X_train.shape[2]
print(input_size)

num_classes = len(np.unique(y_train))
print(num_classes)

In [ ]:
# flatten the image
X_train = np.reshape(X_train, [X_train.shape[0], input_size])
X_train = X_train.astype('float32') / 255

X_test = np.reshape(X_test, [X_test.shape[0], input_size])
X_test = X_test.astype('float32') / 255

* **Machine learning models**

In [ ]:


# Create instances of the models
logistic_regression = LogisticRegression()
decision_tree = DecisionTreeClassifier()
random_forest = RandomForestClassifier()
svm_model = svm.SVC(kernel='rbf')
xgb_model = xgb.XGBClassifier(objective="binary:logistic", random_state=42)
kn_clas = KNeighborsClassifier()
log_reg = LogisticRegression()
nb_model = MultinomialNB()

In [ ]:
# Train the models
logistic_regression.fit(X_train, y_train)


In [ ]:
decision_tree.fit(X_train, y_train)


In [ ]:
random_forest.fit(X_train, y_train)


In [ ]:
xgb_model.fit(X_train, y_train)


In [ ]:
kn_clas.fit(X_train, y_train)


In [ ]:
nb_model.fit(X_train, y_train)


In [ ]:
svm_model.fit(X_train, y_train)

In [ ]:
# Obtain predictions from the models
logistic_regression_preds = logistic_regression.predict(X_test)
decision_tree_preds = decision_tree.predict(X_test)
random_forest_preds = random_forest.predict(X_test)
xgb_model_preds = xgb_model.predict(X_test)
kn_clas_preds = kn_clas.predict(X_test)
nb_model_preds = nb_model.predict(X_test)
svm_model_preds = svm_model.predict(X_test)


In [ ]:
models = ['Logistic Regression', 'Decision Tree', 'Random Forest', 'xgb_model', 'kn_clas' , 'nb_model' , 'svm_model']
predictions = [logistic_regression_preds, decision_tree_preds, random_forest_preds, xgb_model_preds, kn_clas_preds, nb_model_preds , svm_model_preds]


In [ ]:
report_df = pd.DataFrame(columns=['Model', 'Class', 'Precision', 'Recall', 'F1-Score', 'Support'])


In [ ]:
for model, preds in zip(models, predictions):
    report = classification_report(y_test, preds, output_dict=True)
    
    for class_label, metrics in report.items():
        if class_label != 'accuracy':
            report_df = report_df.append({
                'Model': model,
                'Class': class_label,
                'Precision': metrics['precision'],
                'Recall': metrics['recall'],
                'F1-Score': metrics['f1-score'],
                'Support': metrics['support']
            }, ignore_index=True)


In [ ]:
report_df

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Get unique class names and models
class_names = report_df['Class'].unique()
model_names = report_df['Model'].unique()

# Set the number of classes and models
num_classes = len(class_names)
num_models = len(model_names)

# Set the width of each bar
bar_width = 0.8

# Set the positions of the bars on the x-axis
positions = np.arange(num_classes)

# Set the colors for the models
colors = ['red', 'blue', 'green', 'orange', 'purple']

# Create a dictionary to store the F1-Scores for each class
f1_scores_by_class = {class_name: [] for class_name in class_names}

# Iterate over each row in the dataframe
for _, row in report_df.iterrows():
    class_name = row['Class']
    f1_score = row['F1-Score']
    f1_scores_by_class[class_name].append(f1_score)

# Plot the F1-Scores for each class and model using a grouped bar plot
plt.figure(figsize=(10, 6))

# Iterate over each model
for i, model_name in enumerate(model_names):
    # Get the F1-Scores for the current model
    f1_scores = [f1_scores_by_class[class_name][i] for class_name in class_names]
    # Get the color for the current model
    color = colors[i % len(colors)]
    # Calculate the positions for the current model
    current_positions = positions + (i - num_models / 2 + 0.5) * bar_width / num_models
    # Plot the bars for the current model
    plt.bar(current_positions, f1_scores, width=bar_width / num_models, label=model_name, color=color)

plt.xlabel('Class')
plt.ylabel('F1-Score')
plt.title('F1-Score Comparison by Class and Model[SIFT]')
plt.xticks(positions, class_names)
plt.legend(title='Model')

plt.tight_layout()
plt.show()

# Classification using IBP feature extration 

In [ ]:
# Function to extract LBP features from an image
def extract_lbp_features(image):
    gray = np.array(image)
    lbp = local_binary_pattern(gray, 8, 1, method='uniform')
    hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, 60), range=(0, 59))
    return hist


In [ ]:

def process_images():
    features = []
    for filename in tqdm(data['path']):
        if filename.endswith('.jpg') or filename.endswith('.png'):
            img = Image.open(filename)
            img = ImageOps.grayscale(img)
            img = img.resize((64, 64))
            lbp_features = extract_lbp_features(img)
            features.append(lbp_features)
    
    return features


In [ ]:
from skimage.feature import local_binary_pattern


lbp_class_dir = '/kaggle/working/'
lbp_features = process_images()
np.save('lbp_features.npy', lbp_features)

In [ ]:
lbp_features.shape

In [ ]:
label_img = data['result'].map({'Normal': 0, 'COVID': 1, 'Lung_Opacity' : 2,
                               'Viral_Pneumonia' : 3})

print(lbp_features.shape, label_img.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(lbp_features, label_img, 
                                                    test_size=0.2, stratify=label_img)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

In [ ]:
round(y_train.value_counts() / y_train.shape[0] * 100,2)

In [ ]:
# Create instances of the models
logistic_regression = LogisticRegression()
decision_tree = DecisionTreeClassifier()
random_forest = RandomForestClassifier()
svm_model = svm.SVC(kernel='rbf')
xgb_model = xgb.XGBClassifier(objective="binary:logistic", random_state=42)
kn_clas = KNeighborsClassifier()
log_reg = LogisticRegression()
nb_model = MultinomialNB()

In [ ]:
# Train the models
logistic_regression.fit(X_train, y_train)

In [ ]:
decision_tree.fit(X_train, y_train)

In [ ]:
random_forest.fit(X_train, y_train)

In [ ]:
xgb_model.fit(X_train, y_train)

In [ ]:
kn_clas.fit(X_train, y_train)

In [ ]:
nb_model.fit(X_train, y_train)

In [ ]:
svm_model.fit(X_train, y_train)

In [ ]:
# Obtain predictions from the models
logistic_regression_preds = logistic_regression.predict(X_test)
decision_tree_preds = decision_tree.predict(X_test)
random_forest_preds = random_forest.predict(X_test)
xgb_model_preds = xgb_model.predict(X_test)
kn_clas_preds = kn_clas.predict(X_test)
nb_model_preds = nb_model.predict(X_test)
svm_model_preds = svm_model.predict(X_test)

In [ ]:
models = ['Logistic Regression', 'Decision Tree', 'Random Forest', 'xgb_model', 'kn_clas' , 'nb_model' , 'svm_model']
predictions = [logistic_regression_preds, decision_tree_preds, random_forest_preds, xgb_model_preds, kn_clas_preds, nb_model_preds , svm_model_preds]

In [ ]:
report_df = pd.DataFrame(columns=['Model', 'Class', 'Precision', 'Recall', 'F1-Score', 'Support'])

In [ ]:
for model, preds in zip(models, predictions):
    report = classification_report(y_test, preds, output_dict=True)
    
    for class_label, metrics in report.items():
        if class_label != 'accuracy':
            report_df = report_df.append({
                'Model': model,
                'Class': class_label,
                'Precision': metrics['precision'],
                'Recall': metrics['recall'],
                'F1-Score': metrics['f1-score'],
                'Support': metrics['support']
            }, ignore_index=True)

In [ ]:
report_df

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Get unique class names and models
class_names = report_df['Class'].unique()
model_names = report_df['Model'].unique()

# Set the number of classes and models
num_classes = len(class_names)
num_models = len(model_names)

# Set the width of each bar
bar_width = 0.8

# Set the positions of the bars on the x-axis
positions = np.arange(num_classes)

# Set the colors for the models
colors = ['red', 'blue', 'green', 'orange', 'purple']

# Create a dictionary to store the F1-Scores for each class
f1_scores_by_class = {class_name: [] for class_name in class_names}

# Iterate over each row in the dataframe
for _, row in report_df.iterrows():
    class_name = row['Class']
    f1_score = row['F1-Score']
    f1_scores_by_class[class_name].append(f1_score)

# Plot the F1-Scores for each class and model using a grouped bar plot
plt.figure(figsize=(10, 6))

# Iterate over each model
for i, model_name in enumerate(model_names):
    # Get the F1-Scores for the current model
    f1_scores = [f1_scores_by_class[class_name][i] for class_name in class_names]
    # Get the color for the current model
    color = colors[i % len(colors)]
    # Calculate the positions for the current model
    current_positions = positions + (i - num_models / 2 + 0.5) * bar_width / num_models
    # Plot the bars for the current model
    plt.bar(current_positions, f1_scores, width=bar_width / num_models, label=model_name, color=color)

plt.xlabel('Class')
plt.ylabel('F1-Score')
plt.title('F1-Score Comparison by Class and Model')
plt.xticks(positions, class_names)
plt.legend(title='Model')

plt.tight_layout()
plt.show()

# Classification using HOG feature extration

In [ ]:
# Function to extract Hog features from an image
from skimage.feature import hog
def extract_hog_features(image):
    # Convert image to grayscale
    gray = np.array(image)
    
    # Compute HOG features
    features = hog(gray, orientations=9, pixels_per_cell=(8, 8),
                   cells_per_block=(2, 2), transform_sqrt=True,
                   block_norm='L2-Hys')

    return features

In [ ]:
# Function to process all images in a directory and extract SIFT features
def process_images():
    features = []
    for filename in tqdm(data['path']):
        if filename.endswith('.jpg') or filename.endswith('.png'):
            img = Image.open(filename)
            img = ImageOps.grayscale(img)
            img = img.resize((64, 64))
            Hog_features = extract_hog_features(img)
            features.append(Hog_features)
    
    return features

In [ ]:
class_directory = '/kaggle/working/'
class_2_features = process_images()
np.save('class_2_features.npy', class_2_features)

In [ ]:
label_img = data['result'].map({'Normal': 0, 'COVID': 1, 'Lung_Opacity': 2, 'Viral_Pneumonia': 3})


In [ ]:
print(class_2_features.shape, label_img.shape)

In [ ]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(class_2_features, label_img, 
                                                    test_size=0.2, stratify=label_img)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

In [ ]:
round(y_train.value_counts() / y_train.shape[0] * 100,2)

**machine learning model**

In [ ]:
# Create instances of the models
logistic_regression = LogisticRegression()
decision_tree = DecisionTreeClassifier()
random_forest = RandomForestClassifier()
svm_model = svm.SVC(kernel='rbf')
xgb_model = xgb.XGBClassifier(objective="binary:logistic", random_state=42)
kn_clas = KNeighborsClassifier()
log_reg = LogisticRegression()
nb_model = MultinomialNB()

In [ ]:
# Train the models
logistic_regression.fit(X_train, y_train)

In [ ]:
decision_tree.fit(X_train, y_train)


In [ ]:
random_forest.fit(X_train, y_train)


In [ ]:
xgb_model.fit(X_train, y_train)


In [ ]:
kn_clas.fit(X_train, y_train)


In [ ]:
nb_model.fit(X_train, y_train)


In [ ]:
svm_model.fit(X_train, y_train)

In [ ]:
# Obtain predictions from the models
logistic_regression_preds = logistic_regression.predict(X_test)
decision_tree_preds = decision_tree.predict(X_test)
random_forest_preds = random_forest.predict(X_test)
xgb_model_preds = xgb_model.predict(X_test)
kn_clas_preds = kn_clas.predict(X_test)
nb_model_preds = nb_model.predict(X_test)
svm_model_preds = svm_model.predict(X_test)


In [ ]:
models = ['Logistic Regression', 'Decision Tree', 'Random Forest', 'xgb_model', 'kn_clas' , 'nb_model' , 'svm_model']
predictions = [logistic_regression_preds, decision_tree_preds, random_forest_preds, xgb_model_preds, kn_clas_preds, nb_model_preds , svm_model_preds]


In [ ]:
report_df = pd.DataFrame(columns=['Model', 'Class', 'Precision', 'Recall', 'F1-Score', 'Support'])


In [ ]:
for model, preds in zip(models, predictions):
    report = classification_report(y_test, preds, output_dict=True)
    
    for class_label, metrics in report.items():
        if class_label != 'accuracy':
            report_df = report_df.append({
                'Model': model,
                'Class': class_label,
                'Precision': metrics['precision'],
                'Recall': metrics['recall'],
                'F1-Score': metrics['f1-score'],
                'Support': metrics['support']
            }, ignore_index=True)


In [ ]:
report_df


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Get unique class names and models
class_names = report_df['Class'].unique()
model_names = report_df['Model'].unique()

# Set the number of classes and models
num_classes = len(class_names)
num_models = len(model_names)

# Set the width of each bar
bar_width = 0.8

# Set the positions of the bars on the x-axis
positions = np.arange(num_classes)

# Set the colors for the models
colors = ['red', 'blue', 'green', 'orange', 'purple']

# Create a dictionary to store the F1-Scores for each class
f1_scores_by_class = {class_name: [] for class_name in class_names}

# Iterate over each row in the dataframe
for _, row in report_df.iterrows():
    class_name = row['Class']
    f1_score = row['F1-Score']
    f1_scores_by_class[class_name].append(f1_score)

# Plot the F1-Scores for each class and model using a grouped bar plot
plt.figure(figsize=(10, 6))

# Iterate over each model
for i, model_name in enumerate(model_names):
    # Get the F1-Scores for the current model
    f1_scores = [f1_scores_by_class[class_name][i] for class_name in class_names]
    # Get the color for the current model
    color = colors[i % len(colors)]
    # Calculate the positions for the current model
    current_positions = positions + (i - num_models / 2 + 0.5) * bar_width / num_models
    # Plot the bars for the current model
    plt.bar(current_positions, f1_scores, width=bar_width / num_models, label=model_name, color=color)

plt.xlabel('Class')
plt.ylabel('F1-Score')
plt.title('F1-Score Comparison by Class and Model')
plt.xticks(positions, class_names)
plt.legend(title='Model')

plt.tight_layout()
plt.show()